In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


## VAE OOF

In [4]:
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score


# 재현성
np.random.seed(42)
torch.manual_seed(42)


X_array = X_cn7.to_numpy(dtype=np.float32)
y_array = y_cn7.to_numpy()

input_dim = X_array.shape[1]


class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)

        eps = torch.randn_like(std)

        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)

        z = self.reparameterize(
            mu,
            logvar
        )

        reconstructed = self.decoder(z)

        return reconstructed, mu, logvar

## VAE LOSS

In [5]:
def vae_loss(
    reconstructed,
    x,
    mu,
    logvar,
    beta=0.01
):

    reconstruction_loss = torch.mean(
        (reconstructed - x) ** 2
    )

    kl_loss = -0.5 * torch.mean(
        1 + logvar
        - mu.pow(2)
        - logvar.exp()
    )

    return (
        reconstruction_loss
        + beta * kl_loss
    )

## 5-fold OFF

In [6]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_vae_score = np.zeros(
    len(X_array)
)


for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_array, y_array),
    start=1
):

    print(f"\nFold {fold}")

    # fold별 seed 고정
    torch.manual_seed(42 + fold)

    X_train = X_array[train_idx]
    y_train = y_array[train_idx]

    X_val = X_array[val_idx]


    # 정상 데이터만 학습
    X_train_normal = X_train[
        y_train == 0
    ]


    train_tensor = torch.tensor(
        X_train_normal,
        dtype=torch.float32
    )

    train_loader = DataLoader(
        TensorDataset(train_tensor),
        batch_size=64,
        shuffle=True
    )


    model = VAE(
        input_dim=input_dim,
        latent_dim=8
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )


    # 학습
    model.train()

    for epoch in range(100):

        for (batch,) in train_loader:

            optimizer.zero_grad()

            reconstructed, mu, logvar = model(
                batch
            )

            loss = vae_loss(
                reconstructed,
                batch,
                mu,
                logvar,
                beta=0.01
            )

            loss.backward()

            optimizer.step()


    # Validation anomaly score
    model.eval()

    X_val_tensor = torch.tensor(
        X_val,
        dtype=torch.float32
    )


    with torch.no_grad():

        # 평가 시 random sampling 대신
        # latent mean(mu)을 사용
        mu, logvar = model.encode(
            X_val_tensor
        )

        reconstructed = model.decoder(mu)

        reconstruction_error = torch.mean(
            (
                X_val_tensor
                - reconstructed
            ) ** 2,
            dim=1
        )


    oof_vae_score[val_idx] = (
        reconstruction_error.cpu().numpy()
    )


print(
    "\nVAE OOF PR-AUC:",
    average_precision_score(
        y_array,
        oof_vae_score
    )
)


Fold 1

Fold 2

Fold 3

Fold 4

Fold 5

VAE OOF PR-AUC: 0.4231035325996964


In [7]:
vae_df = pd.DataFrame({
    "Target": y_array,
    "VAE_Reconstruction_Error": oof_vae_score
})


vae_df["Rank"] = (
    vae_df[
        "VAE_Reconstruction_Error"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


display(
    vae_df[
        vae_df["Target"] == 1
    ].sort_values("Rank")
)

,Target,VAE_Reconstruction_Error,Rank
118,1,6.551638,1.0
119,1,6.377701,2.0
120,1,6.368419,3.0
115,1,6.152295,4.0
117,1,5.813727,5.0
116,1,5.303996,6.0
95,1,1.111392,9.0
113,1,0.465474,63.0
105,1,0.327020,169.0
103,1,0.267039,275.0


## Threshold 확인

In [8]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

results = []

for top_n in [
    5, 6, 7, 8, 9, 10,
    15, 20, 25, 30,
    50, 75, 100, 150, 200
]:

    threshold = np.sort(oof_vae_score)[-top_n]

    y_pred = (
        oof_vae_score >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_array,
        y_pred
    ).ravel()

    results.append({
        "Top N": top_n,
        "Threshold": threshold,
        "Precision": precision_score(
            y_array,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_array,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_array,
            y_pred,
            zero_division=0
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


vae_threshold_df = pd.DataFrame(results)

display(vae_threshold_df)

,Top N,Threshold,Precision,Recall,F1,TP,FP,FN,TN
0,5,5.813727,1.000000,0.294118,0.454545,5,0,12,1194
1,6,5.303996,1.000000,0.352941,0.521739,6,0,11,1194
2,7,4.482820,0.857143,0.352941,0.500000,6,1,11,1193
3,8,3.087653,0.750000,0.352941,0.480000,6,2,11,1192
4,9,1.111392,0.777778,0.411765,0.538462,7,2,10,1192
5,10,0.960105,0.636364,0.411765,0.500000,7,4,10,1190
6,15,0.743826,0.466667,0.411765,0.437500,7,8,10,1186
7,20,0.653877,0.350000,0.411765,0.378378,7,13,10,1181
8,25,0.599439,0.280000,0.411765,0.333333,7,18,10,1176
9,30,0.564113,0.233333,0.411765,0.297872,7,23,10,1171


In [9]:
from scipy.stats import rankdata

# 점수가 클수록 이상치 → 높은 점수부터 1등
vae_rank = rankdata(
    -oof_vae_score,
    method="min"
)

conflict_defects = [
    47, 80, 95, 99, 101,
    103, 105, 107, 109, 111, 113
]

non_conflict_defects = [
    115, 116, 117, 118, 119, 120
]

defect_indices = np.where(
    y_array == 1
)[0]

vae_defect_result = pd.DataFrame({
    "Defect_Index": defect_indices,

    "Type": [
        "Conflict"
        if idx in conflict_defects
        else "Non-conflict"
        for idx in defect_indices
    ],

    "VAE_Score": oof_vae_score[
        defect_indices
    ],

    "VAE_Rank": vae_rank[
        defect_indices
    ].astype(int)
})

display(
    vae_defect_result.sort_values(
        "VAE_Rank"
    )
)

,Defect_Index,Type,VAE_Score,VAE_Rank
14,118,Non-conflict,6.551638,1
15,119,Non-conflict,6.377701,2
16,120,Non-conflict,6.368419,3
11,115,Non-conflict,6.152295,4
13,117,Non-conflict,5.813727,5
12,116,Non-conflict,5.303996,6
2,95,Conflict,1.111392,9
10,113,Conflict,0.465474,63
6,105,Conflict,0.327020,169
5,103,Conflict,0.267039,275


OOF Feature Reconstruction Error

In [10]:
# ============================================================
# VAE OOF - Feature별 Reconstruction Error 저장
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_vae_score = np.zeros(len(X_array))

# 행 × feature
oof_feature_error = np.zeros(
    (len(X_array), X_array.shape[1])
)


for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_array, y_array),
    start=1
):

    print(f"\nFold {fold}")

    torch.manual_seed(42 + fold)

    X_train = X_array[train_idx]
    y_train = y_array[train_idx]

    X_val = X_array[val_idx]


    # 정상 데이터만 학습
    X_train_normal = X_train[
        y_train == 0
    ]


    train_tensor = torch.tensor(
        X_train_normal,
        dtype=torch.float32
    )

    train_loader = DataLoader(
        TensorDataset(train_tensor),
        batch_size=64,
        shuffle=True
    )


    model = VAE(
        input_dim=input_dim,
        latent_dim=8
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )


    # ========================================================
    # 학습
    # ========================================================

    model.train()

    for epoch in range(100):

        for (batch,) in train_loader:

            optimizer.zero_grad()

            reconstructed, mu, logvar = model(
                batch
            )

            loss = vae_loss(
                reconstructed,
                batch,
                mu,
                logvar,
                beta=0.01
            )

            loss.backward()
            optimizer.step()


    # ========================================================
    # Validation reconstruction
    # ========================================================

    model.eval()

    X_val_tensor = torch.tensor(
        X_val,
        dtype=torch.float32
    )


    with torch.no_grad():

        mu, logvar = model.encode(
            X_val_tensor
        )

        reconstructed = model.decoder(mu)

        # feature별 reconstruction error
        feature_error = (
            X_val_tensor - reconstructed
        ) ** 2


    # 전체 anomaly score
    oof_vae_score[val_idx] = (
        feature_error
        .mean(dim=1)
        .cpu()
        .numpy()
    )

    # ★ feature별 reconstruction error 저장
    oof_feature_error[val_idx] = (
        feature_error
        .cpu()
        .numpy()
    )


print(
    "\nVAE OOF PR-AUC:",
    average_precision_score(
        y_array,
        oof_vae_score
    )
)


Fold 1

Fold 2

Fold 3

Fold 4

Fold 5

VAE OOF PR-AUC: 0.4231035325996964


115~120 각각 Reconstruction Error Top 10

In [11]:
feature_names = X_cn7.columns.to_numpy()

non_conflict_defects = [
    115, 116, 117,
    118, 119, 120
]


for idx in non_conflict_defects:

    errors = oof_feature_error[idx]

    top_idx = np.argsort(
        errors
    )[::-1][:10]


    result = pd.DataFrame({
        "Feature": feature_names[top_idx],
        "Reconstruction_Error": errors[top_idx]
    })


    print(
        f"\n[Defect {idx} - Top 10 Reconstruction Error]"
    )

    display(result)


[Defect 115 - Top 10 Reconstruction Error]


,Feature,Reconstruction_Error
0,Max_Injection_Speed,70.980095
1,Cushion_Position,12.757829
2,Injection_Time,9.425538
3,Filling_Time,9.356090
4,Max_Injection_Pressure,9.106870
5,Max_Switch_Over_Pressure,5.016483
6,Mold_Temperature_4,4.662745
7,Barrel_Temperature_3,4.428738
8,Plasticizing_Position,3.677174
9,Mold_Temperature_3,3.664982



[Defect 116 - Top 10 Reconstruction Error]


,Feature,Reconstruction_Error
0,Max_Injection_Speed,68.881042
1,Max_Switch_Over_Pressure,13.373675
2,Cycle_Time,9.220374
3,Filling_Time,5.945352
4,Injection_Time,4.969574
5,Clamp_Close_Time,4.050671
6,Cushion_Position,3.148109
7,Plasticizing_Position,2.778583
8,Barrel_Temperature_1,1.890937
9,Hopper_Temperature,1.212784



[Defect 117 - Top 10 Reconstruction Error]


...

,Feature,Reconstruction_Error
0,Max_Injection_Speed,77.708603
1,Filling_Time,9.340525
2,Injection_Time,8.541001
3,Hopper_Temperature,7.148492
4,Max_Back_Pressure,7.006791
5,Max_Injection_Pressure,6.985006
6,Average_Back_Pressure,5.548110
7,Plasticizing_Position,4.741590
8,Max_Switch_Over_Pressure,4.707141
9,Clamp_Close_Time,4.424597



[Defect 119 - Top 10 Reconstruction Error]


,Feature,Reconstruction_Error
0,Max_Injection_Speed,70.210350
1,Max_Switch_Over_Pressure,12.630090
2,Cushion_Position,10.282685
3,Injection_Time,9.103670
4,Filling_Time,7.931702
5,Max_Injection_Pressure,5.960385
6,Max_Back_Pressure,5.391952
7,Plasticizing_Position,4.427581
8,Average_Back_Pressure,2.994349
9,Mold_Temperature_4,2.774398



[Defect 120 - Top 10 Reconstruction Error]


,Feature,Reconstruction_Error
0,Max_Injection_Speed,67.016212
1,Max_Switch_Over_Pressure,11.479867
2,Injection_Time,7.950588
3,Filling_Time,6.860680
4,Cushion_Position,6.077413
5,Plasticizing_Position,5.812376
6,Max_Back_Pressure,5.643619
7,Barrel_Temperature_6,5.602376
8,Plasticizing_Time,5.514918
9,Hopper_Temperature,4.039951


In [12]:
normal_feature_error = (
    oof_feature_error[
        y_array == 0
    ]
    .mean(axis=0)
)

non_conflict_error = (
    oof_feature_error[
        [115, 116, 117, 118, 119, 120]
    ]
    .mean(axis=0)
)

compare_df = pd.DataFrame({
    "Feature": X_cn7.columns,

    "Normal_Error":
        normal_feature_error,

    "Defect_Error":
        non_conflict_error
})

compare_df["Error_Ratio"] = (
    compare_df["Defect_Error"]
    / (compare_df["Normal_Error"] + 1e-12)
)

compare_df["Error_Difference"] = (
    compare_df["Defect_Error"]
    - compare_df["Normal_Error"]
)

compare_df = compare_df.sort_values(
    "Error_Ratio",
    ascending=False
).reset_index(drop=True)

display(compare_df.head(15))

,Feature,Normal_Error,Defect_Error,Error_Ratio,Error_Difference
0,Max_Injection_Speed,0.027872,71.713539,2572.935062,71.685667
1,Plasticizing_Position,0.012489,3.952417,316.479829,3.939928
2,Filling_Time,0.052769,7.556391,143.196939,7.503622
3,Injection_Time,0.053165,7.523576,141.512508,7.470411
4,Max_Back_Pressure,0.045600,3.342945,73.309554,3.297345
5,Mold_Temperature_4,0.029606,2.096983,70.828865,2.067376
6,Max_Switch_Over_Pressure,0.163541,10.468865,64.013850,10.305325
7,Plasticizing_Time,0.036430,1.618916,44.438858,1.582486
8,Cushion_Position,0.173592,6.366687,36.676144,6.193095
9,Average_Back_Pressure,0.068622,2.193232,31.961096,2.124610


In [13]:
key_features = [
    "Max_Injection_Speed",
    "Plasticizing_Position",
    "Filling_Time",
    "Injection_Time",
    "Max_Switch_Over_Pressure",
    "Cushion_Position"
]

non_conflict_defects = [
    115, 116, 117,
    118, 119, 120
]


# 비충돌 불량 실제 값
defect_values = X_cn7.loc[
    non_conflict_defects,
    key_features
].copy()

print("[Non-conflict Defect Values]")
display(defect_values)


# 정상 분포와 비교
normal_values = X_cn7.loc[
    y_cn7 == 0,
    key_features
]


compare_value_df = pd.DataFrame({
    "Normal_Mean": normal_values.mean(),
    "Normal_Std": normal_values.std(),
    "Normal_Min": normal_values.min(),
    "Normal_Max": normal_values.max(),

    "Defect_Mean": defect_values.mean(),
    "Defect_Min": defect_values.min(),
    "Defect_Max": defect_values.max()
})


compare_value_df["Mean_Difference"] = (
    compare_value_df["Defect_Mean"]
    - compare_value_df["Normal_Mean"]
)


display(
    compare_value_df.sort_values(
        "Mean_Difference",
        key=abs,
        ascending=False
    )
)

[Non-conflict Defect Values]


,Max_Injection_Speed,Plasticizing_Position,Filling_Time,Injection_Time,Max_Switch_Over_Pressure,Cushion_Position
115,9.166316,1.471608,-4.150108,-3.938052,1.256518,-2.613163
116,9.166316,1.471608,-4.150108,-3.938052,1.256518,-2.613163
117,9.555703,1.471608,-4.150108,-3.938052,1.573231,-2.613163
118,9.555703,1.471608,-4.150108,-3.938052,1.573231,-2.613163
119,9.361006,1.592429,-4.150108,-4.339287,1.889993,-2.613163
120,9.361006,1.592429,-4.150108,-4.339287,1.889993,-2.613163


,Normal_Mean,Normal_Std,Normal_Min,Normal_Max,Defect_Mean,Defect_Min,Defect_Max,Mean_Difference
Max_Injection_Speed,-0.038869,0.751256,-1.541785,8.776929,9.361008,9.166316,9.555703,9.399878
Filling_Time,0.009477,0.955161,-7.776799,2.297327,-4.150108,-4.150108,-4.150108,-4.159585
Injection_Time,0.009360,0.957047,-7.549091,2.080358,-4.071797,-4.339287,-3.938052,-4.081158
Cushion_Position,0.018081,0.987404,-0.711164,6.943503,-2.613163,-2.613163,-2.613163,-2.631243
Max_Switch_Over_Pressure,-0.012054,0.997588,-10.462947,2.206755,1.573248,1.256518,1.889993,1.585301
Plasticizing_Position,-0.021307,0.991308,-2.876962,1.592429,1.511882,1.471608,1.592429,1.533188


In [14]:
conflict_pairs = [
    (47, 48),
    (79, 80),
    (95, 96),
    (99, 100),
    (101, 102),
    (103, 104),
    (105, 106),
    (107, 108),
    (109, 110),
    (111, 112),
    (113, 114)
]

transition_result = []

for idx1, idx2 in conflict_pairs:

    target1 = int(y_cn7.loc[idx1])
    target2 = int(y_cn7.loc[idx2])

    if target1 == 0 and target2 == 1:
        transition = "Normal → Defect"

    elif target1 == 1 and target2 == 0:
        transition = "Defect → Normal"

    else:
        transition = "No Transition"

    transition_result.append({
        "First_Index": idx1,
        "First_Target": target1,
        "Second_Index": idx2,
        "Second_Target": target2,
        "Transition": transition
    })

transition_df = pd.DataFrame(transition_result)

display(transition_df)

print("\n[Transition Count]")
display(
    transition_df["Transition"]
    .value_counts()
    .to_frame("Count")
)

,First_Index,First_Target,Second_Index,Second_Target,Transition
0,47,1,48,0,Defect → Normal
1,79,0,80,1,Normal → Defect
2,95,1,96,0,Defect → Normal
3,99,1,100,0,Defect → Normal
4,101,1,102,0,Defect → Normal
5,103,1,104,0,Defect → Normal
6,105,1,106,0,Defect → Normal
7,107,1,108,0,Defect → Normal
8,109,1,110,0,Defect → Normal
9,111,1,112,0,Defect → Normal



[Transition Count]


,Count
Transition,
Defect → Normal,10
Normal → Defect,1


In [15]:
# 이전 생산 대비 변화량
X_delta = X_cn7.diff()

transition_analysis = []

for idx1, idx2 in conflict_pairs:

    # 첫 번째 행이 해당 상태에 진입할 때의 변화
    first_delta = X_delta.loc[idx1].abs()

    # 두 번째 행은 동일 X가 반복될 때의 변화
    second_delta = X_delta.loc[idx2].abs()

    transition_analysis.append({
        "First_Index": idx1,
        "First_Target": int(y_cn7.loc[idx1]),
        "Second_Index": idx2,
        "Second_Target": int(y_cn7.loc[idx2]),

        "First_Delta_Mean":
            first_delta.mean(),

        "First_Delta_Max":
            first_delta.max(),

        "Second_Delta_Mean":
            second_delta.mean(),

        "Second_Delta_Max":
            second_delta.max(),

        "First_Changed_Features":
            (first_delta > 1e-10).sum(),

        "Second_Changed_Features":
            (second_delta > 1e-10).sum()
    })


transition_delta_df = pd.DataFrame(
    transition_analysis
)

display(transition_delta_df)

,First_Index,First_Target,Second_Index,Second_Target,First_Delta_Mean,First_Delta_Max,Second_Delta_Mean,Second_Delta_Max,First_Changed_Features,Second_Changed_Features
0,47,1,48,0,0.472009,2.467485,0.0,0.0,12,0
1,79,0,80,1,0.499531,2.538292,0.0,0.0,17,0
2,95,1,96,0,0.692854,3.053337,0.0,0.0,13,0
3,99,1,100,0,0.523897,2.538389,0.0,0.0,12,0
4,101,1,102,0,0.513276,2.535883,0.0,0.0,11,0
5,103,1,104,0,0.487799,3.381183,0.0,0.0,14,0
6,105,1,106,0,0.547987,2.154175,0.0,0.0,14,0
7,107,1,108,0,0.746502,4.124845,0.0,0.0,17,0
8,109,1,110,0,0.704805,2.855699,0.0,0.0,16,0
9,111,1,112,0,0.504332,2.732188,0.0,0.0,16,0


In [16]:
# 이전 cycle 대비 변화량
delta = X_cn7.diff()

X_delta_cn7 = X_cn7.copy()

X_delta_cn7["Prev_Delta_Mean"] = (
    delta.abs().mean(axis=1)
)

X_delta_cn7["Prev_Delta_Max"] = (
    delta.abs().max(axis=1)
)

X_delta_cn7["Prev_Changed_Features"] = (
    delta.abs() > 1e-10
).sum(axis=1)

# 첫 번째 행은 이전 cycle이 없으므로 제거
X_delta_cn7 = X_delta_cn7.iloc[1:].copy()
y_delta_cn7 = y_cn7.iloc[1:].copy()

display(
    pd.DataFrame({
        "Normal": X_delta_cn7.loc[
            y_delta_cn7 == 0,
            [
                "Prev_Delta_Mean",
                "Prev_Delta_Max",
                "Prev_Changed_Features"
            ]
        ].mean(),

        "Defect": X_delta_cn7.loc[
            y_delta_cn7 == 1,
            [
                "Prev_Delta_Mean",
                "Prev_Delta_Max",
                "Prev_Changed_Features"
            ]
        ].mean()
    })
)

,Normal,Defect
Prev_Delta_Mean,0.270636,0.505703
Prev_Delta_Max,1.284983,2.565202
Prev_Changed_Features,7.919531,10.882353


In [17]:
# 이전 cycle 대비 변화량
delta = X_cn7.diff()

X_delta_cn7 = X_cn7.copy()

X_delta_cn7["Prev_Delta_Mean"] = delta.abs().mean(axis=1)
X_delta_cn7["Prev_Delta_Max"] = delta.abs().max(axis=1)
X_delta_cn7["Prev_Changed_Features"] = (delta.abs() > 1e-10).sum(axis=1)

# 첫 번째 행은 이전 cycle이 없으므로 제거
X_delta_cn7 = X_delta_cn7.iloc[1:].copy()
y_delta_cn7 = y_cn7.iloc[1:].copy()

In [18]:
X_train, X_test, y_train, y_test = split_data(
    X_delta_cn7,
    y_delta_cn7
)

In [19]:
model_composition(
    X_train,
    X_test,
    y_train,
    y_test
)

[LightGBM] [Info] Number of positive: 14, number of negative: 954
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000463 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 714
[LightGBM] [Info] Number of data points in the train set: 968, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.014463 -> initscore=-4.221606
[LightGBM] [Info] Start training from score -4.221606
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

,Pred Normal,Pred Defect
Actual Normal,239,0
Actual Defect,3,0



[Random Forest]
Accuracy : 0.987603305785124
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.20088566827697263

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,239,0
Actual Defect,3,0



[LightGBM]
Accuracy : 0.987603305785124
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.25555555555555554

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,239,0
Actual Defect,3,0



[XGBoost]
Accuracy : 0.987603305785124
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.4888888888888889

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,239,0
Actual Defect,3,0



[CatBoost]
Accuracy : 0.987603305785124
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.24285714285714285

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,239,0
Actual Defect,3,0
